# T9 — Polynomials, Curve Fitting & Interpolation
### ENGR 50 · Engineering Computational Methods and Applications

---
**How to use this notebook:**  
Run every cell **top to bottom** (Shift+Enter). Each topic goes:  
*concept explained* → *simple example* → *engineering example* → **✏️ Your Turn**

**Topics:**
1. Setup and Imports
2. Polynomials — Representing, Evaluating, Roots
3. Polynomial Arithmetic and Derivatives
4. Curve Fitting with `np.polyfit` — Finding the Best-Fit Equation
5. Residuals — How Good Is the Fit?
6. Choosing the Right Polynomial Degree
7. Exponential Curve Fitting (Linearisation Trick)
8. Linear Interpolation with `np.interp`
9. Cubic-Spline Interpolation with `CubicSpline`
10. Extrapolation — Estimating Outside the Data
11. Engineering Applications (5 worked examples)


---
## 1. Setup — Run This Cell First

We use three libraries throughout this notebook:
- **NumPy** (`np`) — arrays, polynomial math
- **Matplotlib** (`plt`) — plotting
- **SciPy** — cubic spline interpolation

Run this cell once before anything else.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True

print('Libraries loaded. NumPy version:', np.__version__)


---
## 2. Polynomials — Representing, Evaluating, Roots

### What is a polynomial?

A **polynomial** is a sum of terms: each term is a coefficient times x raised to a power.

$$f(x) = a_n x^n + a_{n-1} x^{n-1} + \cdots + a_1 x + a_0$$

**In Python/NumPy**, store a polynomial as an array of coefficients **highest power first**:

```
f(x) = 3x^3 - 2x^2 + x - 5
       [3,   -2,    1,  -5]
        ^     ^     ^   ^
        x^3  x^2   x^1  x^0
```

### Key functions

| Task | Python (NumPy) | MATLAB |
|------|----------------|--------|
| Store polynomial | `p = np.array([3, -2, 1, -5])` | `p = [3 -2 1 -5]` |
| Evaluate at x | `np.polyval(p, x)` | `polyval(p, x)` |
| Find roots | `np.roots(p)` | `roots(p)` |
| Fit data | `np.polyfit(x, y, deg)` | `polyfit(x, y, deg)` |


In [ ]:
# Simple: represent and evaluate a polynomial
# f(x) = 3x^3 - 2x^2 + x - 5

p = np.array([3, -2, 1, -5])   # coefficients, highest power first

x0 = 2.0
y0 = np.polyval(p, x0)
print(f'f({x0}) = {y0}')
print(f'By hand: 3(8) - 2(4) + 2 - 5 = {3*8 - 2*4 + 2 - 5}')

x_vals = np.array([0.0, 1.0, 2.0, 3.0])
y_vals = np.polyval(p, x_vals)
print()
print('x       f(x)')
for x, y in zip(x_vals, y_vals):
    print(f'{x:.1f}     {y:.1f}')


In [ ]:
# Engineering: pressure sensor calibration polynomial
# Lab calibration:  P(V) = 12.3*V^2 - 0.8*V + 2.1  [PSI from Volts]

cal = np.array([12.3, -0.8, 2.1])   # [a2, a1, a0]

voltages  = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
pressures = np.polyval(cal, voltages)

print('Voltage (V)   Pressure (PSI)')
print('-' * 30)
for V, P in zip(voltages, pressures):
    print(f'  {V:.1f}             {P:.2f}')

x_fine = np.linspace(0, 3.5, 200)
y_fine = np.polyval(cal, x_fine)
plt.figure()
plt.plot(x_fine, y_fine, 'b-', linewidth=2, label='Calibration curve')
plt.plot(voltages, pressures, 'ro', markersize=6, label='Spot-check readings')
plt.xlabel('Sensor Voltage (V)'); plt.ylabel('Pressure (PSI)')
plt.title('Pressure Sensor Calibration: P(V) = 12.3V^2 - 0.8V + 2.1')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — polyval
>
> A flow meter has calibration polynomial: **Q(V) = 5.2·V² + 1.3·V − 0.4** [L/min from Volts]
>
> 1. Store the polynomial as a NumPy array (highest power first).
> 2. Use `np.polyval` to find the flow rate at V = 1.0, 1.5, 2.0, 2.5 V.
> 3. Print a table of Voltage vs. Flow Rate.
> 4. Plot the calibration curve from V = 0 to V = 3.
>
> *Expected: at V = 2.0 V you should get Q ≈ 22.8 L/min*


In [ ]:
# YOUR TURN — polyval
# Fill in the blanks

# 1. Store the polynomial Q(V) = 5.2*V^2 + 1.3*V - 0.4
q_cal = np.array([_____, _____, _____])   # [a2, a1, a0]

# 2. Evaluate at these voltages
v_test = np.array([1.0, 1.5, 2.0, 2.5])
q_test = np.polyval(_____, _____)

# 3. Print table
print('Voltage (V)   Flow (L/min)')
for V, Q in zip(v_test, q_test):
    print(f'  {V:.1f}           {Q:.2f}')

# 4. Plot
v_fine = np.linspace(0, 3, 200)
plt.figure()
plt.plot(v_fine, np.polyval(q_cal, v_fine), 'g-', linewidth=2)
plt.xlabel('Voltage (V)'); plt.ylabel('Flow Rate (L/min)')
plt.title('Flow Meter Calibration Curve')
plt.tight_layout(); plt.show()


---
### Polynomial Roots

**Roots** are the x-values where the polynomial equals zero: f(x) = 0.

```
np.roots(p)   ->  finds all roots (may be complex)
```

Engineering use: find when a signal crosses zero, when two curves intersect, when an equation balances.


In [ ]:
# Simple: roots of a quadratic
# x^2 - 2x - 3 = 0  -->  (x-3)(x+1) = 0  -->  roots at x=3 and x=-1

p_quad = np.array([1, -2, -3])
roots = np.roots(p_quad)
print('Quadratic roots:', roots)
print('Check: f(3) =', np.polyval(p_quad, 3))
print('Check: f(-1) =', np.polyval(p_quad, -1))


In [ ]:
# Engineering: find when two temperature profiles cross
# Profile A (cooling): T_A(t) = -0.5*t^2 + 2*t + 80
# Profile B (heating): T_B(t) =  0.3*t^2 +   t + 40

pA = np.array([-0.5, 2.0, 80.0])
pB = np.array([ 0.3, 1.0, 40.0])

p_diff = pA - pB
print('Difference polynomial coefficients:', p_diff)

crossings = np.roots(p_diff)
print('Crossings (t values):', crossings)

real_crossings = [r.real for r in crossings if abs(r.imag) < 1e-9 and r.real > 0]
print('Physical crossing times (t > 0):', [f'{t:.2f} s' for t in real_crossings])

t = np.linspace(0, 12, 200)
plt.figure()
plt.plot(t, np.polyval(pA, t), 'b-', label='Profile A (cooling)')
plt.plot(t, np.polyval(pB, t), 'r-', label='Profile B (heating)')
for tc in real_crossings:
    plt.axvline(tc, color='k', linestyle='--', label=f'Crossing at t={tc:.2f} s')
plt.xlabel('Time (s)'); plt.ylabel('Temperature (C)')
plt.title('Temperature Profile Crossings')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — roots
>
> Two voltage signals in a circuit are modeled as:
> - Signal A: V_A(t) = −t² + 6t + 2
> - Signal B: V_B(t) = t² − 2t + 10
>
> 1. Store each as a NumPy polynomial array.
> 2. Compute the difference polynomial V_A − V_B.
> 3. Use `np.roots` to find when the signals are equal (V_A = V_B).
> 4. Keep only real, positive roots (physically meaningful times).
> 5. Plot both signals from t = 0 to t = 6 and mark the crossing point.
>
> *Hint: They are equal when V_A − V_B = 0*


In [ ]:
# YOUR TURN — roots
# Signal A: V_A(t) = -t^2 + 6t + 2
# Signal B: V_B(t) =  t^2 - 2t + 10

# 1. Store polynomials
pVA = np.array([_____, _____, _____])
pVB = np.array([_____, _____, _____])

# 2. Difference polynomial
p_diff = _____ - _____

# 3. Find roots
crossings = np.roots(_____)
print('All roots:', crossings)

# 4. Keep real, positive roots
real_cross = [r.real for r in crossings if abs(r.imag) < 1e-9 and r.real > 0]
print('Crossing times:', real_cross)

# 5. Plot
t = np.linspace(0, 6, 200)
plt.figure()
plt.plot(t, np.polyval(pVA, t), 'b-', label='Signal A')
plt.plot(t, np.polyval(pVB, t), 'r-', label='Signal B')
for tc in real_cross:
    plt.axvline(tc, color='k', linestyle='--', label=f't = {tc:.2f} s')
plt.xlabel('Time (s)'); plt.ylabel('Voltage (V)')
plt.title('Signal Crossing'); plt.legend(); plt.tight_layout(); plt.show()


---
## 3. Polynomial Arithmetic and Derivatives

| Operation | Python (NumPy) | MATLAB |
|-----------|----------------|--------|
| Multiply two polynomials | `np.polymul(p1, p2)` | `conv(p1, p2)` |
| Add two polynomials | `np.polyadd(p1, p2)` | `p1 + p2` (if same length) |
| Derivative | `np.polyder(p)` | `polyder(p)` |
| Integral | `np.polyint(p)` | `polyint(p)` |

> **Why derivatives?** The derivative of a curve-fit polynomial gives you the *rate of change* — velocity from position, heat flux from temperature, etc.


In [ ]:
# Simple: polynomial multiplication
# (x + 2)(x^2 - x + 3) = x^3 + x^2 + x + 6

p1 = np.array([1, 2])
p2 = np.array([1, -1, 3])

product = np.polymul(p1, p2)
print('Product coefficients:', product)
# Expected: [1, 1, 1, 6]


In [ ]:
# Engineering: derivative of position gives velocity
# Piston position (mm): x(t) = -0.5*t^3 + 3*t^2 + 2*t + 10

pos_poly = np.array([-0.5, 3.0, 2.0, 10.0])
vel_poly = np.polyder(pos_poly)
print('Position polynomial:', pos_poly)
print('Velocity polynomial:', vel_poly)

t_vals = np.array([0.0, 1.0, 2.0, 3.0, 4.0])
pos = np.polyval(pos_poly, t_vals)
vel = np.polyval(vel_poly, t_vals)

print()
print(f"{'Time (s)':<10} {'Position (mm)':<16} {'Velocity (mm/s)'}")
print('-' * 40)
for t, x, v in zip(t_vals, pos, vel):
    print(f'{t:<10.1f} {x:<16.2f} {v:.2f}')

t_fine = np.linspace(0, 4, 200)
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.plot(t_fine, np.polyval(pos_poly, t_fine), 'b-')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Position (mm)'); ax1.set_title('Position x(t)')
ax2.plot(t_fine, np.polyval(vel_poly, t_fine), 'r-')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Velocity (mm/s)'); ax2.set_title('Velocity v(t) = dx/dt')
plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — polyder
>
> A robot arm's angular position (degrees) is modeled as:
> **θ(t) = −2t³ + 9t² + 3t + 0**
>
> 1. Store θ(t) as a NumPy polynomial array.
> 2. Use `np.polyder` to find the angular velocity ω(t) = dθ/dt.
> 3. Use `np.polyder` again on ω(t) to find angular acceleration α(t) = dω/dt.
> 4. Print θ, ω, and α at t = 0, 1, 2, 3 seconds.
> 5. Plot position and velocity side by side from t = 0 to t = 3.


In [ ]:
# YOUR TURN — polyder
# theta(t) = -2t^3 + 9t^2 + 3t + 0

# 1. Store the polynomial
theta_poly = np.array([_____, _____, _____, _____])

# 2. Angular velocity: omega = d(theta)/dt
omega_poly = np.polyder(_____)

# 3. Angular acceleration: alpha = d(omega)/dt
alpha_poly = np.polyder(_____)

print('theta(t):', theta_poly)
print('omega(t):', omega_poly)
print('alpha(t):', alpha_poly)

# 4. Evaluate at t = 0, 1, 2, 3
t_vals = np.array([0., 1., 2., 3.])
print()
print(f"{'t (s)':<8} {'theta (deg)':<14} {'omega (deg/s)':<16} {'alpha (deg/s^2)'}")
print('-' * 55)
for t in t_vals:
    th = np.polyval(theta_poly, t)
    om = np.polyval(omega_poly, t)
    al = np.polyval(alpha_poly, t)
    print(f'{t:<8.0f} {th:<14.2f} {om:<16.2f} {al:.2f}')

# 5. Plot
t_fine = np.linspace(0, 3, 200)
fig, (ax1, ax2) = plt.subplots(1, 2)
ax1.plot(t_fine, np.polyval(theta_poly, t_fine), 'b-')
ax1.set_xlabel('Time (s)'); ax1.set_ylabel('Angle (deg)'); ax1.set_title('Position theta(t)')
ax2.plot(t_fine, np.polyval(omega_poly, t_fine), 'r-')
ax2.set_xlabel('Time (s)'); ax2.set_ylabel('Ang. Velocity (deg/s)'); ax2.set_title('Velocity omega(t)')
plt.tight_layout(); plt.show()


---
## 4. Curve Fitting with `np.polyfit`

### The Big Idea

**Curve fitting** finds the polynomial equation that *best describes the trend* in scattered data.  
It does NOT need to pass through every point — it minimises the total squared error (least squares).

```
coeffs = np.polyfit(x, y, deg)   # deg = degree of polynomial
```

| Degree | Equation | Use when |
|--------|----------|----------|
| 1 | y = mx + b | data looks like a straight line |
| 2 | y = ax² + bx + c | data has one curve (parabola) |
| 3+ | higher curves | more complex shapes |

> **Start with the lowest degree that captures the trend.**


In [ ]:
# Simple: fit a straight line to 5 data points
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

coeffs = np.polyfit(x, y, 1)
print('Coefficients [m, b]:', coeffs)
print(f'Line equation: y = {coeffs[0]:.3f}*x + {coeffs[1]:.3f}')

x_fine = np.linspace(0.5, 5.5, 100)
plt.figure()
plt.plot(x, y, 'ko', markersize=8, label='Data points')
plt.plot(x_fine, np.polyval(coeffs, x_fine), 'b-', label='Linear fit')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Linear Curve Fit')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Engineering: temperature vs. time during motor warm-up
time_min  = np.array([0., 10., 20., 30., 40., 50., 60., 70., 80.])
temp_degC = np.array([22., 35., 50., 64., 74., 83., 89., 94., 97.])

c1 = np.polyfit(time_min, temp_degC, 1)
c2 = np.polyfit(time_min, temp_degC, 2)

print('Linear fit:    y =', f'{c1[0]:.4f}*t + {c1[1]:.4f}')
print('Quadratic fit: y =', f'{c2[0]:.6f}*t^2 + {c2[1]:.4f}*t + {c2[2]:.4f}')

t_fine = np.linspace(-5, 90, 300)
plt.figure()
plt.plot(time_min, temp_degC, 'ko', markersize=8, label='Measured data')
plt.plot(t_fine, np.polyval(c1, t_fine), 'b--', label='Linear fit (deg 1)')
plt.plot(t_fine, np.polyval(c2, t_fine), 'r-',  label='Quadratic fit (deg 2)')
plt.xlabel('Time (min)'); plt.ylabel('Temperature (C)')
plt.title('Motor Warm-Up: Temperature vs. Time')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — polyfit
>
> A load cell measures force vs. displacement during a spring compression test:
>
> | Displacement x (mm) | Force F (N) |
> |----------------------|-------------|
> | 0 | 0 |
> | 2 | 18 |
> | 4 | 41 |
> | 6 | 58 |
> | 8 | 79 |
> | 10 | 102 |
>
> 1. Store x and F as NumPy arrays.
> 2. Fit a **degree-1** polynomial (Hooke's Law: F = k·x).
> 3. Print the spring constant k (the slope).
> 4. Predict the force at x = 5 mm and x = 9 mm.
> 5. Plot the data and the fit on the same axes.


In [ ]:
# YOUR TURN — polyfit
x_mm = np.array([0., 2., 4., 6., 8., 10.])
F_N  = np.array([0., 18., 41., 58., 79., 102.])

# 2. Fit degree-1 polynomial
c = np.polyfit(_____, _____, _____)
k = c[0]   # spring constant (N/mm)
print(f'Spring constant k = {k:.2f} N/mm')

# 4. Predict at x = 5 and x = 9
for x_pred in [5.0, 9.0]:
    F_pred = np.polyval(c, x_pred)
    print(f'Predicted F at x={x_pred} mm: {F_pred:.1f} N')

# 5. Plot
x_fine = np.linspace(0, 11, 200)
plt.figure()
plt.plot(x_mm, F_N, 'ko', markersize=8, label='Test data')
plt.plot(x_fine, np.polyval(c, x_fine), 'b-', linewidth=2,
         label=f'Linear fit: F = {k:.2f}*x')
plt.xlabel('Displacement (mm)'); plt.ylabel('Force (N)')
plt.title('Spring Compression Test')
plt.legend(); plt.tight_layout(); plt.show()


---
## 5. Residuals — How Good Is the Fit?

A **residual** is the difference between measured and predicted:

```
residual_i = y_measured_i - y_fit_i
```

**R² (R-squared)** summarises fit quality in one number:
- R² = 1.0 → perfect fit
- R² ≥ 0.99 → excellent
- R² < 0.8 → poor fit, try a different degree or model


In [ ]:
# Simple: compute and plot residuals
x = np.array([1., 2., 3., 4., 5.])
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])

coeffs = np.polyfit(x, y, 1)
y_fit  = np.polyval(coeffs, x)
resid  = y - y_fit

print('Residuals:', resid.round(3))
ss_res = np.sum(resid**2)
ss_tot = np.sum((y - y.mean())**2)
r2 = 1 - ss_res / ss_tot
print(f'R^2 = {r2:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
x_fine = np.linspace(0.5, 5.5, 100)
ax1.plot(x, y, 'ko', markersize=8, label='Data')
ax1.plot(x_fine, np.polyval(coeffs, x_fine), 'b-', label='Fit')
for xi, yi, ri in zip(x, y, resid):
    ax1.plot([xi, xi], [yi, yi-ri], 'r-', linewidth=2)
ax1.set_title(f'Fit (R^2 = {r2:.4f})'); ax1.legend()
ax2.stem(x, resid); ax2.axhline(0, color='k')
ax2.set_xlabel('x'); ax2.set_ylabel('Residual')
ax2.set_title('Residuals (random = good fit)')
plt.tight_layout(); plt.show()


In [ ]:
# Engineering: compare residuals for linear vs. quadratic fit
time_min  = np.array([0., 10., 20., 30., 40., 50., 60., 70., 80.])
temp_degC = np.array([22., 35., 50., 64., 74., 83., 89., 94., 97.])

c1 = np.polyfit(time_min, temp_degC, 1)
c2 = np.polyfit(time_min, temp_degC, 2)
r1 = temp_degC - np.polyval(c1, time_min)
r2 = temp_degC - np.polyval(c2, time_min)

def r_squared(y, y_fit):
    ss_res = np.sum((y - y_fit)**2)
    ss_tot = np.sum((y - y.mean())**2)
    return 1 - ss_res / ss_tot

r2_1 = r_squared(temp_degC, np.polyval(c1, time_min))
r2_2 = r_squared(temp_degC, np.polyval(c2, time_min))

print(f'Linear fit    R^2 = {r2_1:.4f}   max residual = {abs(r1).max():.2f} C')
print(f'Quadratic fit R^2 = {r2_2:.4f}   max residual = {abs(r2).max():.2f} C')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.stem(time_min, r1, linefmt='b-', markerfmt='bo')
ax1.axhline(0, color='k')
ax1.set_title(f'Linear residuals  (R^2={r2_1:.3f})')
ax1.set_xlabel('Time (min)'); ax1.set_ylabel('Residual (C)')
ax2.stem(time_min, r2, linefmt='r-', markerfmt='ro')
ax2.axhline(0, color='k')
ax2.set_title(f'Quadratic residuals  (R^2={r2_2:.3f})')
ax2.set_xlabel('Time (min)'); ax2.set_ylabel('Residual (C)')
plt.tight_layout(); plt.show()
print('Observation: linear residuals show a curved PATTERN -> degree 2 fits better.')


> ### ✏️ Your Turn — residuals and R²
>
> Use the spring data from the previous section:
>
> | x (mm) | F (N) |
> |---------|-------|
> | 0, 2, 4, 6, 8, 10 | 0, 18, 41, 58, 79, 102 |
>
> 1. Compute the residuals for your degree-1 fit.
> 2. Compute R² for the degree-1 fit.
> 3. Now fit a **degree-2** polynomial and compute its R².
> 4. Plot the residuals for both fits side by side.
> 5. Which degree is more appropriate? Why?


In [ ]:
# YOUR TURN — residuals
x_mm = np.array([0., 2., 4., 6., 8., 10.])
F_N  = np.array([0., 18., 41., 58., 79., 102.])

# Fit degree 1 and degree 2
c1 = np.polyfit(x_mm, F_N, 1)
c2 = np.polyfit(x_mm, F_N, 2)

# Residuals
resid1 = F_N - np.polyval(c1, x_mm)
resid2 = F_N - np.polyval(c2, x_mm)

# R^2
def r_squared(y, y_fit):
    return 1 - np.sum((y - y_fit)**2) / np.sum((y - y.mean())**2)

r2_1 = r_squared(F_N, np.polyval(c1, x_mm))
r2_2 = r_squared(F_N, np.polyval(c2, x_mm))

print(f'Degree 1: R^2 = {r2_1:.4f}')
print(f'Degree 2: R^2 = {r2_2:.4f}')

# Plot residuals side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.stem(x_mm, resid1, linefmt='b-', markerfmt='bo')
ax1.axhline(0, color='k')
ax1.set_title(f'Deg 1 residuals  R^2={r2_1:.3f}')
ax1.set_xlabel('x (mm)'); ax1.set_ylabel('Residual (N)')

ax2.stem(x_mm, resid2, linefmt='r-', markerfmt='ro')
ax2.axhline(0, color='k')
ax2.set_title(f'Deg 2 residuals  R^2={r2_2:.3f}')
ax2.set_xlabel('x (mm)'); ax2.set_ylabel('Residual (N)')

plt.tight_layout(); plt.show()
# Which is better? Write your conclusion here:
# ANSWER:


---
## 6. Choosing the Right Polynomial Degree

Higher degree ≠ better fit.

**Overfitting** — the polynomial chases noise, not the real trend. Fits the data you have perfectly but fails on new data.

**Rule of thumb:**
- Start with degree 1 or 2
- Increase degree if residuals show a systematic pattern
- Stop when residuals look random OR R² ≈ 1
- **Never use degree ≥ number of data points**


In [ ]:
# Overfitting demo: degree 1, 2, and 6 on 7 data points
x7 = np.array([0.9, 1.5, 3.0, 4.0, 6.0, 8.0, 9.5])
y7 = np.array([0.9, 1.5, 2.5, 3.0, 3.8, 4.5, 4.8])

x_fine = np.linspace(0.5, 10, 300)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, deg, color in zip(axes, [1, 2, 6], ['b', 'g', 'r']):
    c = np.polyfit(x7, y7, deg)
    y_fit = np.polyval(c, x7)
    r2 = 1 - np.sum((y7-y_fit)**2) / np.sum((y7-y7.mean())**2)
    ax.plot(x7, y7, 'ko', markersize=8, label='Data')
    ax.plot(x_fine, np.polyval(c, x_fine), color=color, linewidth=2,
            label=f'Deg {deg}  R^2={r2:.3f}')
    ax.set_ylim(-1, 7)
    ax.set_title(f'Degree {deg}  (R^2={r2:.3f})')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.legend(fontsize=8); ax.grid(True)

plt.suptitle('Underfitting vs. Good fit vs. Overfitting')
plt.tight_layout(); plt.show()
print('Degree 2 captures the trend without chasing noise.')
print('Degree 6 passes through every point but wiggles wildly between them.')


> ### ✏️ Your Turn — choosing degree
>
> A civil engineer measures bridge deflection (mm) at different loads (kN):
>
> | Load (kN) | Deflection (mm) |
> |-----------|-----------------|
> | 10 | 2.1 |
> | 20 | 4.3 |
> | 30 | 6.8 |
> | 40 | 9.0 |
> | 50 | 11.4 |
> | 60 | 14.2 |
> | 70 | 16.5 |
>
> 1. Fit degree-1, degree-2, and degree-3 polynomials.
> 2. Compute R² for each.
> 3. Plot all three fits on the same axes.
> 4. Which degree would you choose and why?


In [ ]:
# YOUR TURN — choosing degree
load_kN  = np.array([10., 20., 30., 40., 50., 60., 70.])
defl_mm  = np.array([2.1, 4.3, 6.8, 9.0, 11.4, 14.2, 16.5])

x_fine = np.linspace(5, 75, 300)
plt.figure()
plt.plot(load_kN, defl_mm, 'ko', markersize=10, label='Data', zorder=5)

for deg, color, ls in zip([1, 2, 3], ['b', 'g', 'r'], ['-', '--', ':']):
    c = np.polyfit(load_kN, defl_mm, deg)
    y_fit = np.polyval(c, load_kN)
    r2 = 1 - np.sum((defl_mm - y_fit)**2) / np.sum((defl_mm - defl_mm.mean())**2)
    print(f'Degree {deg}: R^2 = {r2:.4f}')
    plt.plot(x_fine, np.polyval(c, x_fine), color=color, linestyle=ls,
             linewidth=2, label=f'Deg {deg}  R^2={r2:.3f}')

plt.xlabel('Load (kN)'); plt.ylabel('Deflection (mm)')
plt.title('Bridge Deflection: Choosing Polynomial Degree')
plt.legend(); plt.tight_layout(); plt.show()

# ANSWER: Which degree is best and why?
# ANSWER:


---
## 7. Exponential Curve Fitting — The Linearisation Trick

Model: **y = b · eᵐˣ**

`np.polyfit` fits polynomials only — but taking **ln** of both sides gives:

```
ln(y) = ln(b) + m·x    ← this is linear!
```

Steps:
1. `ln_y = np.log(y)`
2. `coeffs = np.polyfit(x, ln_y, 1)`
3. `m = coeffs[0]`,  `b = np.exp(coeffs[1])`

Only works when **all y > 0**.


In [ ]:
# Simple: fit y = b * exp(m*x)
x_exp = np.array([0., 1., 2., 3., 4., 5.])
y_exp = np.array([2.0, 3.3, 5.4, 8.9, 14.7, 24.2])

ln_y   = np.log(y_exp)
coeffs = np.polyfit(x_exp, ln_y, 1)
m = coeffs[0]
b = np.exp(coeffs[1])
print(f'Fitted model: y = {b:.3f} * exp({m:.3f} * x)')

x_fine = np.linspace(-0.2, 5.5, 200)
y_fine = b * np.exp(m * x_fine)
plt.figure()
plt.plot(x_exp, y_exp, 'ko', markersize=8, label='Data')
plt.plot(x_fine, y_fine, 'r-', linewidth=2, label=f'Fit: y = {b:.2f}*exp({m:.2f}*x)')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Exponential Curve Fit')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Engineering: bacterial growth — find doubling time
time_min = np.array([0., 20., 40., 60., 80., 100., 120.])
OD       = np.array([0.05, 0.09, 0.17, 0.31, 0.58, 1.07, 1.98])

ln_OD  = np.log(OD)
coeffs = np.polyfit(time_min, ln_OD, 1)
k      = coeffs[0]
OD0    = np.exp(coeffs[1])
doubling_time = np.log(2) / k

print(f'Fitted model : OD(t) = {OD0:.4f} * exp({k:.4f} * t)')
print(f'Doubling time: {doubling_time:.1f} minutes')
print(f'Predicted OD at t=150 min: {OD0 * np.exp(k * 150):.3f}')

t_fine = np.linspace(0, 150, 300)
plt.figure()
plt.plot(time_min, OD, 'ko', markersize=8, label='Measured OD')
plt.plot(t_fine, OD0 * np.exp(k * t_fine), 'g-', linewidth=2,
         label=f'Exp fit (doubling={doubling_time:.0f} min)')
plt.xlabel('Time (min)'); plt.ylabel('Optical Density')
plt.title('Bacterial Growth: Exponential Curve Fit')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — exponential fit
>
> A capacitor discharges through a resistor. Voltage vs. time data:
>
> | Time (s) | Voltage (V) |
> |----------|-------------|
> | 0 | 9.0 |
> | 1 | 6.1 |
> | 2 | 4.1 |
> | 3 | 2.7 |
> | 4 | 1.8 |
> | 5 | 1.2 |
>
> The model is **V(t) = V₀ · e^(−t/τ)**
>
> 1. Linearise by taking `np.log(V)`.
> 2. Fit a degree-1 polynomial to `(time, ln_V)`.
> 3. Extract V₀ = exp(coeffs[1]) and τ = −1/coeffs[0].
> 4. Print V₀ and τ. (Expected: V₀ ≈ 9 V, τ ≈ 2.5 s)
> 5. Plot the data and fitted curve from t = 0 to t = 6.


In [ ]:
# YOUR TURN — exponential fit
time_s = np.array([0., 1., 2., 3., 4., 5.])
volt_V = np.array([9.0, 6.1, 4.1, 2.7, 1.8, 1.2])

# 1. Take ln
ln_V = np.log(_____)

# 2. Fit degree-1 polynomial
coeffs = np.polyfit(_____, _____, 1)

# 3. Extract V0 and tau
V0  = np.exp(coeffs[1])
tau = -1.0 / coeffs[0]

print(f'V0  = {V0:.2f} V')
print(f'tau = {tau:.2f} s')

# 5. Plot
t_fine = np.linspace(0, 6, 200)
V_fit  = V0 * np.exp(-t_fine / tau)

plt.figure()
plt.plot(time_s, volt_V, 'ko', markersize=8, label='Measured')
plt.plot(t_fine, V_fit, 'b-', linewidth=2, label=f'Fit: V0={V0:.1f}V, tau={tau:.1f}s')
plt.xlabel('Time (s)'); plt.ylabel('Voltage (V)')
plt.title('Capacitor Discharge')
plt.legend(); plt.tight_layout(); plt.show()


---
## 8. Linear Interpolation — `np.interp`

**Interpolation** estimates values *between* known data points and passes through every point exactly.

```python
yi = np.interp(xi, x_data, y_data)
```

- Connects adjacent points with straight lines
- Returns the boundary value outside the range (no extrapolation)
- Best for: lookup tables, non-smooth data


In [ ]:
# Simple: interpolate between 5 known points
x_data = np.array([0., 1., 2., 3., 4.])
y_data = np.array([0., 1., 4., 9., 16.])

xi_values = np.array([0.5, 1.5, 2.5, 3.5])
yi_values = np.interp(xi_values, x_data, y_data)

print('xi      yi (interpolated)   y_exact (x^2)')
for xi, yi in zip(xi_values, yi_values):
    print(f'{xi:.1f}     {yi:.2f}                {xi**2:.2f}')

plt.figure()
plt.plot(x_data, y_data, 'ko', markersize=10, label='Known data')
plt.plot(x_data, y_data, 'k--', linewidth=1, alpha=0.4)
plt.plot(xi_values, yi_values, 'r^', markersize=10, label='Interpolated')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Linear Interpolation with np.interp')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Engineering: Type-K thermocouple lookup table
voltage_mV = np.array([0.00, 2.02, 4.10, 6.13, 8.14, 10.15, 12.13, 14.11, 16.04])
temp_C     = np.array([  0.,  50., 100., 150., 200.,  250.,  300.,  350.,  400.])

field_mV = np.array([1.5, 5.2, 9.0, 11.0, 15.6])
field_C  = np.interp(field_mV, voltage_mV, temp_C)

print('Thermocouple Readings')
print(f"{'Voltage (mV)':<15} {'Temperature (C)'}")
print('-' * 32)
for mV, C in zip(field_mV, field_C):
    print(f'{mV:<15.1f} {C:.1f}')

mV_fine = np.linspace(0, 16, 300)
C_fine  = np.interp(mV_fine, voltage_mV, temp_C)
plt.figure()
plt.plot(voltage_mV, temp_C, 'ks', markersize=8, label='Datasheet table')
plt.plot(mV_fine, C_fine, 'b-', linewidth=1.5, label='Linear interpolation')
plt.plot(field_mV, field_C, 'r^', markersize=10, label='Field readings')
plt.xlabel('Thermocouple Voltage (mV)'); plt.ylabel('Temperature (C)')
plt.title('Type-K Thermocouple Lookup')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — np.interp
>
> A pump efficiency table (from manufacturer datasheet):
>
> | Flow (GPM) | Efficiency (%) |
> |------------|----------------|
> | 0 | 0 |
> | 20 | 45 |
> | 40 | 68 |
> | 60 | 78 |
> | 80 | 72 |
> | 100 | 55 |
> | 120 | 30 |
>
> 1. Store flow and efficiency as NumPy arrays.
> 2. Use `np.interp` to look up efficiency at 30, 50, 70, and 90 GPM.
> 3. Print a table of those results.
> 4. Plot the datasheet values and the interpolated points.
> 5. At what flow rate (from your table) is the pump most efficient?


In [ ]:
# YOUR TURN — np.interp
flow_gpm = np.array([0., 20., 40., 60., 80., 100., 120.])
eff_pct  = np.array([0., 45., 68., 78., 72.,  55.,  30.])

# 2. Interpolate at these operating points
query_flow = np.array([30., 50., 70., 90.])
query_eff  = np.interp(_____, _____, _____)

# 3. Print table
print(f"{'Flow (GPM)':<13} {'Efficiency (%)'}")
print('-' * 28)
for Q, e in zip(query_flow, query_eff):
    print(f'{Q:<13.0f} {e:.1f}')

# 4. Plot
flow_fine = np.linspace(0, 120, 300)
eff_fine  = np.interp(flow_fine, flow_gpm, eff_pct)
plt.figure()
plt.plot(flow_gpm, eff_pct, 'ks', markersize=8, label='Datasheet')
plt.plot(flow_fine, eff_fine, 'b-', label='Interpolated curve')
plt.plot(query_flow, query_eff, 'r^', markersize=10, label='Operating points')
plt.xlabel('Flow Rate (GPM)'); plt.ylabel('Efficiency (%)')
plt.title('Pump Efficiency Curve')
plt.legend(); plt.tight_layout(); plt.show()

# 5. Peak efficiency
best_idx = np.argmax(eff_pct)
print(f'Peak efficiency {eff_pct[best_idx]}% at {flow_gpm[best_idx]} GPM')


---
## 9. Cubic-Spline Interpolation — Smooth Curves Through Every Point

Linear interpolation creates **sharp corners**. A **cubic spline** uses a separate 3rd-degree polynomial between each pair of points, ensuring smooth joins (matching slope and curvature).

```python
from scipy.interpolate import CubicSpline
cs = CubicSpline(x_data, y_data)   # build once
yi = cs(xi)                         # evaluate anywhere
```

Use `CubicSpline` when the physical quantity is smooth (temperature, deflection, velocity).  
Use `np.interp` when you want no smoothing (lookup tables).


In [ ]:
# Simple: compare linear vs. cubic spline
x_data = np.array([0., 1., 2., 3., 4., 5.])
y_data = np.array([0., 0.8, 0.9, 0.1, -0.8, -1.0])

cs = CubicSpline(x_data, y_data)
x_fine = np.linspace(0, 5, 300)

plt.figure()
plt.plot(x_data, y_data, 'ko', markersize=10, label='Data points', zorder=5)
plt.plot(x_fine, np.interp(x_fine, x_data, y_data), 'b--', linewidth=2,
         label='Linear interp (sharp corners)')
plt.plot(x_fine, cs(x_fine), 'r-', linewidth=2, label='Cubic spline (smooth)')
plt.xlabel('x'); plt.ylabel('y')
plt.title('Linear vs. Cubic Spline Interpolation')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Engineering: pipe velocity profile and flow rate
r = np.array([0., 1., 2., 3., 4., 5., 6.])   # cm from centre
v = np.array([3.0, 2.9, 2.7, 2.3, 1.7, 0.8, 0.0])   # m/s

cs_v = CubicSpline(r, v)
r_fine = np.linspace(0, 6, 300)
v_fine = cs_v(r_fine)

Q = cs_v.integrate(0, 6) * 2 * np.pi
print(f'Flow rate Q = {Q:.4f}  m/s·cm^2  =  {Q*1e-4:.6f}  m^3/s')

plt.figure()
plt.plot(r, v, 'ko', markersize=10, label='Measured', zorder=5)
plt.plot(r_fine, v_fine, 'b-', linewidth=2, label='Cubic spline')
plt.fill_between(r_fine, v_fine, alpha=0.2, label='Area ~ Q')
plt.xlabel('Radius (cm)'); plt.ylabel('Velocity (m/s)')
plt.title('Pipe Velocity Profile (Cubic Spline)')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — CubicSpline
>
> A structural beam deflection is measured at 6 points along its length:
>
> | Position x (m) | Deflection y (mm) |
> |-----------------|-------------------|
> | 0 | 0.0 |
> | 1 | 3.2 |
> | 2 | 8.5 |
> | 3 | 10.1 |
> | 4 | 7.8 |
> | 5 | 0.0 |
>
> 1. Build a `CubicSpline` through these points.
> 2. Find the maximum deflection and where it occurs (use `np.argmax` on a fine grid).
> 3. Estimate the deflection at x = 2.5 m and x = 3.7 m.
> 4. Plot the data points and the spline curve.
> 5. Also plot the linear interpolation on the same axes for comparison.


In [ ]:
# YOUR TURN — CubicSpline
x_pos = np.array([0., 1., 2., 3., 4., 5.])
y_def = np.array([0.0, 3.2, 8.5, 10.1, 7.8, 0.0])

# 1. Build cubic spline
cs_beam = CubicSpline(_____, _____)

# 2. Find maximum deflection
x_fine = np.linspace(0, 5, 500)
y_fine = cs_beam(x_fine)

max_idx  = np.argmax(y_fine)
max_x    = x_fine[max_idx]
max_defl = y_fine[max_idx]
print(f'Maximum deflection: {max_defl:.2f} mm at x = {max_x:.2f} m')

# 3. Estimate at specific positions
for x_query in [2.5, 3.7]:
    print(f'Deflection at x={x_query} m: {cs_beam(x_query):.2f} mm')

# 4 & 5. Plot spline and linear interpolation
plt.figure()
plt.plot(x_pos, y_def, 'ko', markersize=10, label='Measured', zorder=5)
plt.plot(x_fine, np.interp(x_fine, x_pos, y_def), 'b--', linewidth=2, label='Linear interp')
plt.plot(x_fine, y_fine, 'r-', linewidth=2, label='Cubic spline')
plt.plot(max_x, max_defl, 'r*', markersize=14, label=f'Max = {max_defl:.1f} mm at {max_x:.1f} m')
plt.xlabel('Position (m)'); plt.ylabel('Deflection (mm)')
plt.title('Beam Deflection Profile')
plt.legend(); plt.tight_layout(); plt.show()


---
## 10. Extrapolation — Estimating Outside the Data

| Method | Outside range behaviour |
|--------|------------------------|
| `np.interp` | Clamps to boundary value — NO extrapolation |
| `CubicSpline` | Continues end polynomial — can diverge quickly |
| `np.polyval` (fitted) | Continues polynomial — reasonable for small steps |

> **Golden rule:** extrapolating more than ~10–20% beyond your data range is risky. Always state uncertainty.


In [ ]:
# Compare extrapolation of all three methods
x_data = np.array([2., 4., 6., 8., 10., 12.])
y_data = np.array([63., 70., 77., 82., 86., 89.])

c2 = np.polyfit(x_data, y_data, 2)
cs = CubicSpline(x_data, y_data)
x_ext = np.linspace(0, 16, 300)

plt.figure(figsize=(10, 4))
plt.plot(x_data, y_data, 'ko', markersize=10, label='Data', zorder=5)
plt.plot(x_ext, np.interp(x_ext, x_data, y_data), 'b-',  linewidth=2, label='np.interp (clamps)')
plt.plot(x_ext, np.polyval(c2, x_ext),              'g--', linewidth=2, label='polyfit deg 2')
plt.plot(x_ext, cs(x_ext),                           'r:', linewidth=2, label='CubicSpline')
plt.axvline(x_data[0],  color='gray', linestyle='--', alpha=0.5)
plt.axvline(x_data[-1], color='gray', linestyle='--', alpha=0.5, label='Data range')
plt.xlabel('Hour of Day'); plt.ylabel('Temperature (C)')
plt.title('Extrapolation: Three Methods Compared')
plt.legend(fontsize=9); plt.tight_layout(); plt.show()
print('np.interp: safe but flat outside the range.')
print('polyfit:   reasonable small extrapolation for smooth data.')
print('CubicSpline: can diverge quickly outside the data range.')


> ### ✏️ Your Turn — extrapolation
>
> Use the beam deflection data from section 9:
>
> | x (m) | y (mm) |
> |--------|--------|
> | 0, 1, 2, 3, 4, 5 | 0.0, 3.2, 8.5, 10.1, 7.8, 0.0 |
>
> 1. Fit a degree-4 polynomial to the data.
> 2. Build a `CubicSpline` through the data.
> 3. Plot both methods from x = −1 to x = 6 (beyond the data range).
> 4. Also show `np.interp` on the same plot.
> 5. Which method behaves most reasonably outside the data? Why?


In [ ]:
# YOUR TURN — extrapolation
x_pos = np.array([0., 1., 2., 3., 4., 5.])
y_def = np.array([0.0, 3.2, 8.5, 10.1, 7.8, 0.0])

# Fit polynomial and spline
c4      = np.polyfit(x_pos, y_def, 4)
cs_beam = CubicSpline(x_pos, y_def)

# Extended range (beyond data)
x_ext = np.linspace(-1, 6, 300)

plt.figure(figsize=(10, 4))
plt.plot(x_pos, y_def, 'ko', markersize=10, label='Data', zorder=5)
plt.plot(x_ext, np.interp(x_ext, x_pos, y_def), 'b-',  linewidth=2, label='np.interp')
plt.plot(x_ext, np.polyval(c4, x_ext),            'g--', linewidth=2, label='polyfit deg 4')
plt.plot(x_ext, cs_beam(x_ext),                    'r:', linewidth=2, label='CubicSpline')
plt.axvline(x_pos[0],  color='gray', linestyle='--', alpha=0.5)
plt.axvline(x_pos[-1], color='gray', linestyle='--', alpha=0.5, label='Data range')
plt.xlabel('Position (m)'); plt.ylabel('Deflection (mm)')
plt.title('Beam Deflection: Extrapolation Comparison')
plt.legend(fontsize=9); plt.tight_layout(); plt.show()

# ANSWER: Which method is most reasonable outside the data and why?
# ANSWER:


---
## 11. Engineering Applications — Putting It All Together

Five complete worked examples. Each ends with a **Your Turn** challenge on the same scenario.


---
### Application 1 — Pump Curve: Fit and Predict


In [ ]:
speed_rpm = np.array([1000., 1500., 2000., 2500., 3000.])
flow_gpm  = np.array([  12.,   25.,   45.,   72.,  105.])

c = np.polyfit(speed_rpm, flow_gpm, 2)
print(f'Pump curve: Q = {c[0]:.2e}*N^2 + {c[1]:.4f}*N + {c[2]:.3f}')

query_speeds = np.array([1200., 1750., 2250., 2800.])
query_flows  = np.polyval(c, query_speeds)

print()
print(f"{'Speed (RPM)':<14} {'Flow (GPM)'}")
print('-' * 28)
for N, Q in zip(query_speeds, query_flows):
    print(f'{N:<14.0f} {Q:.1f}')

y_fit = np.polyval(c, speed_rpm)
r2 = 1 - np.sum((flow_gpm-y_fit)**2) / np.sum((flow_gpm-flow_gpm.mean())**2)
print(f'R^2 = {r2:.4f}')

N_fine = np.linspace(800, 3200, 300)
plt.figure()
plt.plot(speed_rpm, flow_gpm, 'ko', markersize=10, label='Test data')
plt.plot(N_fine, np.polyval(c, N_fine), 'b-', linewidth=2, label=f'Polynomial fit (R^2={r2:.3f})')
plt.plot(query_speeds, query_flows, 'r^', markersize=10, label='Predicted')
plt.xlabel('Speed (RPM)'); plt.ylabel('Flow Rate (GPM)')
plt.title('Pump Performance Curve')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — Pump Curve
>
> The same pump has a **head pressure** (ft) vs. speed curve measured at:
>
> | Speed (RPM) | Head (ft) |
> |-------------|-----------|
> | 1000 | 45 |
> | 1500 | 95 |
> | 2000 | 165 |
> | 2500 | 255 |
> | 3000 | 365 |
>
> 1. Fit a degree-2 polynomial to Head vs. Speed.
> 2. Print the curve equation.
> 3. Predict head pressure at 1800 RPM and 2700 RPM.
> 4. Plot the data and fit. Label axes.


In [ ]:
# YOUR TURN — Pump Head Curve
speed_rpm = np.array([1000., 1500., 2000., 2500., 3000.])
head_ft   = np.array([  45.,   95.,  165.,  255.,  365.])

# 1. Fit degree-2 polynomial
c_head = np.polyfit(_____, _____, _____)
print(f'Head curve: H = {c_head[0]:.2e}*N^2 + {c_head[1]:.4f}*N + {c_head[2]:.3f}')

# 3. Predict
for N in [1800., 2700.]:
    H = np.polyval(c_head, N)
    print(f'At {N:.0f} RPM: Head = {H:.1f} ft')

# 4. Plot
N_fine = np.linspace(800, 3200, 300)
plt.figure()
plt.plot(speed_rpm, head_ft, 'ko', markersize=10, label='Test data')
plt.plot(N_fine, np.polyval(c_head, N_fine), 'b-', linewidth=2, label='Polynomial fit')
plt.xlabel('Speed (RPM)'); plt.ylabel('Head Pressure (ft)')
plt.title('Pump Head Curve')
plt.legend(); plt.tight_layout(); plt.show()


---
### Application 2 — Material Stress-Strain: Find Elastic Modulus


In [ ]:
strain    = np.array([0.000, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.008, 0.010])
stress_MP = np.array([  0.,  210.,  420.,  630.,  836.,  980., 1000., 1010., 1020.])

elastic_idx = strain < 0.004
c_elastic   = np.polyfit(strain[elastic_idx], stress_MP[elastic_idx], 1)
E_modulus   = c_elastic[0]

print(f'Elastic modulus E = {E_modulus:.0f} MPa  ({E_modulus/1000:.1f} GPa)')
print(f'(Typical steel: 200 GPa)')

plt.figure()
plt.plot(strain, stress_MP, 'ko-', markersize=8, label='Test data')
strain_fine = np.linspace(0, 0.004, 100)
plt.plot(strain_fine, np.polyval(c_elastic, strain_fine), 'r--', linewidth=2,
         label=f'Linear fit (E = {E_modulus/1000:.1f} GPa)')
plt.axvline(0.004, color='gray', linestyle=':', label='End of elastic region')
plt.xlabel('Strain'); plt.ylabel('Stress (MPa)')
plt.title('Steel Stress-Strain: Finding Elastic Modulus')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — Stress-Strain
>
> An aluminum specimen has these stress-strain measurements. The elastic region is strain < 0.003:
>
> | Strain | Stress (MPa) |
> |--------|--------------|
> | 0.000 | 0 |
> | 0.001 | 69 |
> | 0.002 | 138 |
> | 0.003 | 200 |
> | 0.004 | 230 |
> | 0.006 | 245 |
> | 0.008 | 250 |
>
> 1. Store strain and stress as arrays.
> 2. Filter to the elastic region (strain < 0.003).
> 3. Fit a line through the elastic region.
> 4. Extract the elastic modulus E (slope in MPa). Expected: ~69 GPa for aluminum.
> 5. Plot the full data + the linear elastic fit.


In [ ]:
# YOUR TURN — Elastic Modulus (Aluminum)
strain_al = np.array([0.000, 0.001, 0.002, 0.003, 0.004, 0.006, 0.008])
stress_al = np.array([  0.,   69.,  138.,  200.,  230.,  245.,  250.])

# 2. Filter to elastic region
elastic = strain_al < 0.003
c_al    = np.polyfit(strain_al[elastic], stress_al[elastic], 1)
E_al    = c_al[0]
print(f'Aluminum E = {E_al:.0f} MPa  ({E_al/1000:.1f} GPa)')

# 5. Plot
plt.figure()
plt.plot(strain_al, stress_al, 'ko-', markersize=8, label='Test data')
s_fine = np.linspace(0, 0.003, 100)
plt.plot(s_fine, np.polyval(c_al, s_fine), 'r--', linewidth=2,
         label=f'Elastic fit (E={E_al/1000:.0f} GPa)')
plt.axvline(0.003, color='gray', linestyle=':', label='End of elastic region')
plt.xlabel('Strain'); plt.ylabel('Stress (MPa)')
plt.title('Aluminum Stress-Strain')
plt.legend(); plt.tight_layout(); plt.show()


---
### Application 3 — Valve Flow Coefficient: Smooth Lookup


In [ ]:
pct_open = np.array([  0.,  10.,  20.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.])
Cv       = np.array([  0.,  0.1,  0.5,  1.8,  4.2,  8.0, 13.5, 20.0, 27.0, 34.0, 40.0])

cs_valve  = CubicSpline(pct_open, Cv)
positions = np.arange(0, 101, 1)
Cv_fine   = cs_valve(positions)

target_Cv  = 15.0
idx        = np.searchsorted(Cv_fine, target_Cv)
pct_target = positions[idx]
print(f'To get Cv = {target_Cv}, open valve to approximately {pct_target}%')

plt.figure()
plt.plot(pct_open, Cv, 'ko', markersize=10, label='Datasheet values', zorder=5)
plt.plot(positions, Cv_fine, 'b-', linewidth=2, label='Cubic spline')
plt.axhline(target_Cv, color='r', linestyle='--', label=f'Target Cv={target_Cv}')
plt.axvline(pct_target, color='r', linestyle=':')
plt.xlabel('Valve Opening (%)'); plt.ylabel('Flow Coefficient Cv')
plt.title('Control Valve Cv Curve (Cubic Spline)')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — Valve Lookup
>
> Using the same valve spline:
>
> 1. Find the valve opening needed to achieve Cv = 5, Cv = 25, and Cv = 35.
> 2. Print a table: Target Cv → Required Opening (%).
> 3. What opening gives the maximum rate of change of Cv with position?
>    (*Hint: evaluate `cs_valve(positions, 1)` to get the derivative — the 1 means first derivative.*)


In [ ]:
# YOUR TURN — Valve Lookup
pct_open = np.array([  0.,  10.,  20.,  30.,  40.,  50.,  60.,  70.,  80.,  90., 100.])
Cv       = np.array([  0.,  0.1,  0.5,  1.8,  4.2,  8.0, 13.5, 20.0, 27.0, 34.0, 40.0])
cs_valve  = CubicSpline(pct_open, Cv)
positions = np.arange(0, 101, 1)
Cv_fine   = cs_valve(positions)

# 1 & 2. Find opening for each target Cv
print(f"{'Target Cv':<12} {'Opening (%)'}")
print('-' * 25)
for target in [5.0, 25.0, 35.0]:
    idx = np.searchsorted(Cv_fine, target)
    print(f'{target:<12.1f} {positions[idx]}')

# 3. Maximum rate of change (derivative)
dCv = cs_valve(positions, 1)   # first derivative
max_rate_idx = np.argmax(dCv)
print(f'Max dCv/d(pct) = {dCv[max_rate_idx]:.3f} at {positions[max_rate_idx]}% opening')


---
### Application 4 — RC Circuit Discharge: Exponential Fit


In [ ]:
time_ms = np.array([0., 5., 10., 15., 20., 25., 30., 35., 40.])
volt_V  = np.array([5.0, 3.6, 2.7, 1.9, 1.4, 1.0, 0.75, 0.55, 0.40])

ln_V   = np.log(volt_V)
coeffs = np.polyfit(time_ms, ln_V, 1)
V0     = np.exp(coeffs[1])
tau    = -1.0 / coeffs[0]

print(f'Initial voltage V0 = {V0:.3f} V  (measured: {volt_V[0]:.1f} V)')
print(f'Time constant tau  = {tau:.2f} ms')
print(f'If C = 100 uF, then R = tau/C = {tau/0.1:.1f} Ohm')

t_fine = np.linspace(0, 45, 300)
V_fit  = V0 * np.exp(-t_fine / tau)
plt.figure()
plt.plot(time_ms, volt_V, 'ko', markersize=8, label='Measured')
plt.plot(t_fine, V_fit, 'b-', linewidth=2, label=f'Exp fit: tau = {tau:.1f} ms')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (V)')
plt.title('RC Circuit Discharge: Exponential Fit')
plt.legend(); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — RC Circuit
>
> A different RC circuit gives this discharge data:
>
> | Time (ms) | Voltage (V) |
> |-----------|-------------|
> | 0 | 12.0 |
> | 10 | 7.9 |
> | 20 | 5.3 |
> | 30 | 3.5 |
> | 40 | 2.3 |
> | 50 | 1.5 |
>
> 1. Fit an exponential model V(t) = V₀ · e^(−t/τ).
> 2. Find V₀ and τ.
> 3. If C = 220 µF, calculate R = τ/C (convert τ from ms to seconds first).
> 4. How long until voltage drops below 1 V? Solve: 1 = V₀ · e^(−t/τ) for t.
> 5. Plot the data and fitted curve.


In [ ]:
# YOUR TURN — RC Circuit
time_ms2 = np.array([0., 10., 20., 30., 40., 50.])
volt_V2  = np.array([12.0, 7.9, 5.3, 3.5, 2.3, 1.5])

# 1. Fit exponential
ln_V2   = np.log(volt_V2)
coeffs2 = np.polyfit(time_ms2, ln_V2, 1)
V0_2    = np.exp(coeffs2[1])
tau_2   = -1.0 / coeffs2[0]   # ms

print(f'V0  = {V0_2:.2f} V')
print(f'tau = {tau_2:.2f} ms')

# 3. Calculate R
C_uF = 220.        # microfarads
tau_s = tau_2 / 1000.   # convert ms to s
R_ohm = tau_s / (C_uF * 1e-6)
print(f'R   = {R_ohm:.1f} Ohm')

# 4. Time until V < 1 V: t = -tau * ln(1/V0)
t_1V = -tau_2 * np.log(1.0 / V0_2)
print(f'Time to drop below 1 V: {t_1V:.1f} ms')

# 5. Plot
t_fine2 = np.linspace(0, 70, 300)
V_fit2  = V0_2 * np.exp(-t_fine2 / tau_2)
plt.figure()
plt.plot(time_ms2, volt_V2, 'ko', markersize=8, label='Measured')
plt.plot(t_fine2, V_fit2, 'b-', linewidth=2, label=f'Fit: tau={tau_2:.1f} ms')
plt.axhline(1.0, color='r', linestyle='--', label='1 V threshold')
plt.axvline(t_1V, color='r', linestyle=':')
plt.xlabel('Time (ms)'); plt.ylabel('Voltage (V)')
plt.title('RC Circuit Discharge')
plt.legend(); plt.tight_layout(); plt.show()


---
### Application 5 — HVAC Daily Temperature Model


In [ ]:
hour   = np.array([0., 2., 4., 6., 8., 10., 12., 14., 16., 18., 20., 22., 24.])
temp_C = np.array([14., 12., 11., 12., 16., 21., 26., 29., 28., 24., 20., 17., 14.])

c4     = np.polyfit(hour, temp_C, 4)
h_fine = np.linspace(0, 24, 1441)
T_fit  = np.polyval(c4, h_fine)

peak_idx  = np.argmax(T_fit)
peak_hour = h_fine[peak_idx]
peak_temp = T_fit[peak_idx]
print(f'Peak temperature: {peak_temp:.1f} C at {peak_hour:.1f} h ({int(peak_hour)}:{int((peak_hour%1)*60):02d})')

above_25 = h_fine[T_fit > 25.0]
if len(above_25) > 0:
    print(f'Hours above 25 C: {above_25[-1]-above_25[0]:.1f} h ({above_25[0]:.1f} to {above_25[-1]:.1f})')

plt.figure()
plt.plot(hour, temp_C, 'ko', markersize=10, label='Hourly data', zorder=5)
plt.plot(h_fine, T_fit, 'b-', linewidth=2, label='Polynomial fit (deg 4)')
plt.axhline(25, color='r', linestyle='--', alpha=0.7, label='Cooling threshold (25 C)')
plt.plot(peak_hour, peak_temp, 'r*', markersize=16,
         label=f'Peak: {peak_temp:.1f} C at {peak_hour:.1f}h')
plt.xlabel('Hour of Day (0=midnight)'); plt.ylabel('Temperature (C)')
plt.title('Daily Temperature Model — HVAC Load Analysis')
plt.legend(fontsize=9); plt.tight_layout(); plt.show()


> ### ✏️ Your Turn — HVAC Analysis
>
> Using the same temperature model from the example above:
>
> 1. Estimate the temperature at 6:30 AM, 11:45 AM, and 8:20 PM.
> 2. Find the **minimum** temperature and when it occurs.
> 3. How many hours is it **below 15°C** (heating needed)?
> 4. If cooling turns on above 25°C and costs \$0.15/hour, calculate the daily cooling cost.


In [ ]:
# YOUR TURN — HVAC Analysis
hour   = np.array([0., 2., 4., 6., 8., 10., 12., 14., 16., 18., 20., 22., 24.])
temp_C = np.array([14., 12., 11., 12., 16., 21., 26., 29., 28., 24., 20., 17., 14.])

c4     = np.polyfit(hour, temp_C, 4)
h_fine = np.linspace(0, 24, 1441)   # every minute
T_fit  = np.polyval(c4, h_fine)

# 1. Temperature at specific times (convert to decimal hours first)
query_times = [6 + 30/60, 11 + 45/60, 20 + 20/60]
labels      = ['6:30 AM', '11:45 AM', '8:20 PM']
print('Temperature estimates:')
for h, label in zip(query_times, labels):
    T = np.polyval(c4, h)
    print(f'  {label}: {T:.1f} C')

# 2. Minimum temperature
min_idx   = np.argmin(T_fit)
min_hour  = h_fine[min_idx]
min_temp  = T_fit[min_idx]
print(f'Minimum: {min_temp:.1f} C at {min_hour:.1f} h ({int(min_hour)}:{int((min_hour%1)*60):02d})')

# 3. Hours below 15 C
below_15 = h_fine[T_fit < 15.0]
hours_below_15 = len(below_15) / 60.0   # each step is 1 minute = 1/60 hour
print(f'Hours below 15 C: {hours_below_15:.1f} h')

# 4. Cooling cost
above_25      = h_fine[T_fit > 25.0]
hours_cooling = len(above_25) / 60.0
cost_per_hour = 0.15
daily_cost    = hours_cooling * cost_per_hour
print(f'Hours above 25 C: {hours_cooling:.1f} h -> Daily cooling cost: ${daily_cost:.2f}')


---
## Quick Reference Card

| Task | Python Code |
|------|-------------|
| Store polynomial | `p = np.array([a_n, ..., a_1, a_0])` |
| Evaluate polynomial | `np.polyval(p, x)` |
| Find roots | `np.roots(p)` |
| Fit polynomial to data | `c = np.polyfit(x, y, deg)` |
| Derivative | `dp = np.polyder(p)` |
| Multiply polynomials | `np.polymul(p1, p2)` |
| Linear interpolation | `np.interp(xi, x_data, y_data)` |
| Cubic spline (build) | `cs = CubicSpline(x_data, y_data)` |
| Cubic spline (evaluate) | `cs(xi)` |
| Spline derivative | `cs(xi, 1)` |
| Spline integral | `cs.integrate(a, b)` |
| Exponential fit | fit `np.log(y)` vs `x` with `polyfit(x, ln_y, 1)` |
| R² | `1 - sum((y-y_fit)^2) / sum((y-mean(y))^2)` |

### Choosing the right tool

| Situation | Use |
|-----------|-----|
| Find the equation behind your data | `np.polyfit` |
| Must pass through every point (lookup table) | `np.interp` |
| Must pass through every point (smooth data) | `CubicSpline` |
| Data follows exponential growth/decay | Linearise with `np.log`, then `polyfit` |
| Estimating outside the data range | `np.polyval` with caution |
